# Identity billiards: a synthetic face bouncing inside its match region

We generate a fixed anchor, move through a slice of latent space, and reflect when ArcFace cosine similarity **falls below** a threshold. The contour is empirical: it can be anisotropic, irregular, or interrupted by face-detection failures. All identities here originate from generated faces.

Run cells in order using the **Face Boundary (StyleGAN + InsightFace)** kernel. The initial map and video take a few minutes on a GPU. See `experiments/README.md` for setup and interpretation.


In [ ]:
from pathlib import Path
import sys
import os
ROOT = Path.cwd()
if not (ROOT / "legacy.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "legacy.py").exists():
    raise RuntimeError("Start Jupyter from the StyleGAN3 root or notebooks directory")
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "outputs" / ".matplotlib"))
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Video
from experiments.face_boundary import *
torch.set_num_threads(4)
print("Python:", sys.executable)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name())


## Choose one reproducible experiment

The default fixed center is the generator's stored learned W mean, `mapping.w_avg`; its rendered face is the reference. Start with shared W and two principal axes in empirical standard-deviation units. `SPACE='z'` instead uses the Gaussian prior mean, zero, which is a different latent point. `BASIS='random'` rotates standardized W axes. `CENTER='seed'` explicitly enables a seeded center. The buffalo_l default is cosine similarity >= 0.45, equivalent to cosine distance <= 0.55; it follows upstream defaults rather than a locally calibrated false-match rate. See `experiments/README.md` for the threshold survey and sources.


In [ ]:
CENTER = "mean"
SEED = 23
SPACE = "w"
BASIS = "pca"
THRESHOLD = DEFAULT_IDENTITY_SIMILARITY
GRID_SIZE = 21
EXTENT = 3.0
FRAMES = 240
FPS = 24
STEP_SIZE = 0.07
center_tag = "mean" if CENTER == "mean" else f"seed{SEED}"
OUTPUT = ROOT / "outputs" / f"notebook-{SPACE}-{center_tag}-tau{THRESHOLD:g}"
OUTPUT.mkdir(parents=True, exist_ok=True)
network = download_network()
generator = load_generator(network)
matcher = load_matcher()
latent = LatentSlice(generator, seed=SEED, space=SPACE, basis_kind=BASIS, center=CENTER)
oracle = IdentityOracle(latent, matcher)
display(__import__("PIL").Image.fromarray(oracle.reference))
print("Anchor self-similarity:", oracle(np.zeros(2)).score)
print(f"Match boundary: cosine similarity >= {THRESHOLD:.2f}; distance <= {1-THRESHOLD:.2f}")


## Map the identity region

Each image is independently detected and aligned using five landmarks. Blank map cells are invalid detections, not evidence of low identity similarity. The white star is the fixed anchor. A 2D contour is a slice through the high-dimensional region, not its projection.


In [ ]:
grid = similarity_grid(oracle, extent=EXTENT, size=GRID_SIZE)
figure = plot_map(grid, THRESHOLD)
display(figure)
plt.close(figure)
print("Invalid cells:", int(np.count_nonzero(grid["reasons"] != "ok")))


## Measure radii rather than assume a circle

Scan rays until the first rejected observation, then refine by bisection. A radius-limit result is censored: we have only a lower bound. A detection exit must be kept separate from an identity exit. Finer scan increments help find thin rejected bands.


In [ ]:
angles = np.linspace(0, 2*np.pi, 32, endpoint=False)
directions = np.column_stack([np.cos(angles), np.sin(angles)])
rays = radial_scan(oracle, directions, THRESHOLD, increment=0.2, max_radius=6.0)
radii = np.array([item["radius"] for item in rays])
identity = np.array([item["reason"] == "identity" for item in rays])
figure, axes = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(6, 5))
axes.plot(np.append(angles, angles[0]), np.append(radii, radii[0]), color="gray")
axes.scatter(angles[identity], radii[identity], label="Identity exit")
axes.scatter(angles[~identity], radii[~identity], marker="x", label="Detection exit / censored")
axes.set_title("First-exit radius in this latent slice")
axes.legend(loc="upper right", bbox_to_anchor=(1.45, 1.2))
plt.show()
if np.count_nonzero(identity) >= 2:
    print("Max/min observed identity-exit radius:", radii[identity].max()/radii[identity].min())
print({reason: sum(item["reason"] == reason for item in rays) for reason in {item["reason"] for item in rays}})


## Let the ball move

Specular reflection uses a finite-difference estimate of the local similarity normal. Every accepted frame must satisfy the threshold. Substeps reduce the chance of skipping a thin rejected band. Invalid detections and the outer radius guard generate separately labeled collisions. Increase `substeps` and reduce `STEP_SIZE` for a stability check.


In [ ]:
trace = billiard(oracle, threshold=THRESHOLD, frames=FRAMES, step_size=STEP_SIZE,
                 substeps=4, gradient_epsilon=0.03, seed=11)
print("Minimum similarity:", trace["scores"].min())
print("Collision events:", len(trace["events"]))
print("Collision reasons:", {reason: sum(event["reason"] == reason for event in trace["events"])
                            for reason in {event["reason"] for event in trace["events"]}})
print("Fallbacks:", sum(event["reflection"] != "normal" for event in trace["events"]))
figure = plot_map(grid, THRESHOLD, trace["positions"])
figure.savefig(OUTPUT / "identity_map.png", dpi=160)
display(figure)
plt.close(figure)


## How many different faces?

A continuous region has infinitely many latent points. Report a finite sample budget and a diversity resolution. Here we greedily select frames at least a chosen **embedding cosine distance** from all previously selected representatives. This count depends on cutoff, order, and path length. Two frames can both match the anchor without matching one another.


In [ ]:
summary = diversity_summary(trace["embeddings"], distance=0.05)
print(summary)
indices = summary["representative_indices"][:12]
figure, axes = plt.subplots(1, len(indices), figsize=(max(3, 2*len(indices)), 2.8), squeeze=False)
for axis, index in zip(axes[0], indices):
    axis.imshow(latent.render(trace["positions"][index])[0])
    axis.set_title(f'frame {index}\ncos {trace["scores"][index]:.2f}')
    axis.axis("off")
plt.show()


## Save the experiment and make an MP4

The video shows the current face, fixed reference, contour, and moving ball. Saved arrays include the exact latent basis, coordinates, scores, and embeddings. Metadata records hashes and repository revisions. The FFmpeg executable is bundled with `imageio-ffmpeg`.


In [ ]:
save_run(OUTPUT, oracle, trace, network, grid=grid,
         extra={"rays": rays, "ray_directions": directions.tolist(), "fps": FPS})
video_path = write_video(OUTPUT / "billiard.mp4", oracle, trace, grid, fps=FPS)
display(Video(str(video_path), embed=True, width=960))
print(OUTPUT)


Next: open **02_geometry_and_controls.ipynb** to sweep thresholds, inspect the local identity metric, and probe higher-dimensional slices. For sharper or more varied videos, increase frames and the map resolution after checking collision diagnostics.
